In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ali = "data_annotated_60_ali.json"
annotated_wilder = "data_annotated_60_wilder.json"

with open(annotated_ali, 'r', encoding='utf-8') as file:
    ali = json.load(file)


with open(annotated_wilder, 'r', encoding='utf-8') as file:
    wilder = json.load(file)

In [2]:
TYPES = ["Lexical", "Syntactic", "Semantic", "Vagueness", "Incompleteness", "Referential"]

for t in TYPES:
    a = [ali[i][t] for i in range(len(ali))]
    w = [wilder[i][t] for i in range(len(wilder))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ali)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} kappa={k:6.3f}  agreement={agree:.2%} ali_pos={sum(a):2d}  wilder_pos={sum(w):2d}")

Lexical         kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0
Syntactic       kappa=-0.034  agreement=92.50% ali_pos= 2  wilder_pos= 1
Semantic        kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0
Vagueness       kappa= 0.429  agreement=80.00% ali_pos= 8  wilder_pos=10
Incompleteness  kappa= 0.310  agreement=80.00% ali_pos= 6  wilder_pos= 8
Referential     kappa=-0.034  agreement=92.50% ali_pos= 2  wilder_pos= 1


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect

In [3]:
# align by id instead of trusting file order
ali_by_id = {r["id"]: r for r in ali}
wilder_by_id = {r["id"]: r for r in wilder}
ids = sorted(set(ali_by_id) & set(wilder_by_id))
disagreements = {t: [i for i in ids if ali_by_id[i][t] != wilder_by_id[i][t]] for t in TYPES}
disagreements

{'Lexical': [],
 'Syntactic': [24, 40, 46],
 'Semantic': [],
 'Vagueness': [35, 36, 37, 40, 41, 42, 43, 60],
 'Incompleteness': [24, 28, 29, 40, 46, 53, 54, 59],
 'Referential': [21, 40, 45]}

In [4]:
for i in ids:
    diffs = [t for t in TYPES if bool(ali_by_id[i][t]) != bool(wilder_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ali_by_id[i]['description']}")
    for t in diffs:
        who = "Ali" if ali_by_id[i][t] else "Wilder"
        print(f"  - {t}: True by {who}")
    for name, rec in (("Ali", ali_by_id[i]), ("Wilder", wilder_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 21
requirement: The system shall allow a personal representative the same level of access as a person over their PHI.
  - Referential: True by Wilder
  note (Ali): their refers to person or personal representeative

id: 24
requirement: The system shall support allowing an unemancipated minor to act as the individual and restrict a parent or guardian’s access to protected health information (PHI) when the minor has legal authority over a specific health care service.
  - Syntactic: True by Wilder
  - Incompleteness: True by Wilder

id: 28
requirement: The system must treat an authorization as invalid if the expiration date has passed or if the expiration event has occurred.
  - Incompleteness: True by Ali
  note (Ali): what is expiration event

id: 29
requirement: The system must treat an authorization as invalid if any authorization element has not been filled out completely.
  - Incompleteness: True by Ali
  note (Ali): authorization is not clear

id: 35
requirement: The system ma